# GAN-A Phase 1: Adversarial Inpainting Training (Linux / Cloud GPU)

Trains the UNet generator + PatchGAN discriminator on paired (masked → healthy) images.

**Architecture:** UNet generator (4ch in → 3ch out) + 70×70 PatchGAN discriminator (7ch in)

**Loss:** λ_adv·Adversarial + λ_L1·L1 + λ_SSIM·SSIM + λ_perc·Perceptual(VGG16)

**Training:** Alternating G/D updates with LSGAN objective, mixed-precision (AMP)

- Images are resized to 512×512
- Saves checkpoint every N epochs with full G+D state
- Comprehensive metric tracking: G_loss, D_loss, L1, SSIM, perceptual, adversarial
- Visual sample outputs saved every N epochs

**Target:** A100 GPU on Lambda Labs cloud

In [ ]:
# 1. Imports
import os
import sys
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')  # headless backend for cloud/SSH
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# 2. Import models and losses
from pathlib import Path
import importlib

# Resolve repo root (notebook is at src/gan/ganA_train_linux.ipynb)
repo_root = Path.cwd().resolve()
# If running from src/gan/, go up two levels
if repo_root.name == 'gan':
    repo_root = repo_root.parents[1]
elif repo_root.name == 'src':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f'Repo root: {repo_root}')

import src.models.unet_inpaint as unet_mod
import src.models.patchgan_discriminator as disc_mod
import src.models.losses as loss_mod
importlib.reload(unet_mod)
importlib.reload(disc_mod)
importlib.reload(loss_mod)

UNet = unet_mod.UNet
PatchGANDiscriminator = disc_mod.PatchGANDiscriminator
GANLoss = loss_mod.GANLoss
PerceptualLoss = loss_mod.PerceptualLoss
SSIMLoss = loss_mod.SSIMLoss
print('Generator, Discriminator, and loss modules loaded.')

In [ ]:
# =============================================================
# 3. CONFIGURATION — edit these to switch datasets / hyperparams
# =============================================================

# ----- Data paths (relative to repo root) -----
DATA_ROOT = os.path.join(str(repo_root), 'data', 'gan_a', 'phase_1')
TRAIN_IMAGE_DIR   = os.path.join(DATA_ROOT, 'train', 'image')
TRAIN_MASK_DIR    = os.path.join(DATA_ROOT, 'train', 'mask')
TRAIN_OUTLINE_DIR = os.path.join(DATA_ROOT, 'train', 'outline')
TEST_IMAGE_DIR    = os.path.join(DATA_ROOT, 'test', 'image')
TEST_MASK_DIR     = os.path.join(DATA_ROOT, 'test', 'mask')
TEST_OUTLINE_DIR  = os.path.join(DATA_ROOT, 'test', 'outline')

# ----- Output directories -----
CHECKPOINT_DIR   = os.path.join(str(repo_root), 'src', 'models', 'checkpoints')
LOG_DIR          = os.path.join(str(repo_root), 'results', 'logs')
SAMPLE_DIR       = os.path.join(str(repo_root), 'results', 'figures', 'training_samples')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

# ----- Resume from checkpoint (set to None to train from scratch) -----
RESUME_CHECKPOINT = None  # e.g. os.path.join(CHECKPOINT_DIR, 'ganA_phase1_epoch10.pth')

# ----- Hyperparameters (tuned for A100 80GB) -----
BATCH_SIZE    = 16     # A100 can handle 16+ at 512x512
NUM_EPOCHS    = 100    # total epochs (includes already-completed ones)
IMAGE_SIZE    = 512    # UNet requires power-of-2 spatial dims
SAVE_EVERY    = 5      # save checkpoint every N epochs
SAMPLE_EVERY  = 5      # save visual samples every N epochs
NUM_WORKERS   = 4      # dataloader workers (Linux supports fork-based multiprocessing)

# ----- Generator -----
G_LEARNING_RATE = 1e-4
G_FEATURES      = [64, 128, 256, 512]  # UNet encoder features

# ----- Discriminator -----
D_LEARNING_RATE  = 2e-4
D_BASE_FEATURES  = 64
D_N_LAYERS       = 3

# ----- Loss weights -----
LAMBDA_L1         = 100.0   # L1 reconstruction
LAMBDA_SSIM       = 10.0    # structural similarity
LAMBDA_PERCEPTUAL = 10.0    # VGG perceptual
LAMBDA_ADV        = 1.0     # adversarial (LSGAN)
GAN_MODE          = 'lsgan' # 'lsgan' or 'vanilla'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    print(f'GPU memory reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB')
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
if RESUME_CHECKPOINT:
    print(f'Will resume from: {RESUME_CHECKPOINT}')
else:
    print('Training from scratch.')
print(f'Image size: {IMAGE_SIZE}x{IMAGE_SIZE}  |  Batch size: {BATCH_SIZE}')
print(f'Loss weights — L1: {LAMBDA_L1}  SSIM: {LAMBDA_SSIM}  Perceptual: {LAMBDA_PERCEPTUAL}  Adversarial: {LAMBDA_ADV}')

In [ ]:
# 4. Dataset class — loads paired (image, outline, target) and resizes to IMAGE_SIZE

class PairedInpaintDataset(Dataset):
    """Loads matched (masked_image, outline, target_image) triplets for inpainting.
    Returns a 4-channel input (RGB + outline) and 3-channel target (RGB)."""
    def __init__(self, image_folder, mask_folder, outline_folder, image_size=512):
        self.images   = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.masks    = sorted([f for f in os.listdir(mask_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.outlines = sorted([f for f in os.listdir(outline_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        assert self.images == self.masks == self.outlines, 'Image, mask, and outline filenames must match!'
        self.image_folder = image_folder
        self.mask_folder = mask_folder
        self.outline_folder = outline_folder
        self.rgb_transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
        self.gray_transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),  # produces [1, H, W] for grayscale
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img     = Image.open(os.path.join(self.image_folder, self.images[index])).convert('RGB')
        mask    = Image.open(os.path.join(self.mask_folder, self.masks[index])).convert('RGB')
        outline = Image.open(os.path.join(self.outline_folder, self.outlines[index])).convert('L')
        img_t     = self.rgb_transform(img)      # [3, H, W]
        mask_t    = self.rgb_transform(mask)      # [3, H, W]
        outline_t = self.gray_transform(outline)  # [1, H, W]
        # Concatenate RGB + outline -> [4, H, W]
        input_4ch = torch.cat([img_t, outline_t], dim=0)
        return input_4ch, mask_t, self.images[index]

# Create datasets and dataloaders
train_dataset = PairedInpaintDataset(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, TRAIN_OUTLINE_DIR, IMAGE_SIZE)
test_dataset  = PairedInpaintDataset(TEST_IMAGE_DIR, TEST_MASK_DIR, TEST_OUTLINE_DIR, IMAGE_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}')

In [ ]:
# 5. Generator, Discriminator, optimizers, losses

# --- Generator ---
netG = UNet(in_channels=4, out_channels=3, features=G_FEATURES).to(device)

# --- Discriminator (input = 4ch model input + 3ch image = 7ch) ---
netD = PatchGANDiscriminator(in_channels=7, base_features=D_BASE_FEATURES, n_layers=D_N_LAYERS).to(device)

# --- Optimizers (beta1=0.5 is standard for GANs) ---
optG = optim.Adam(netG.parameters(), lr=G_LEARNING_RATE, betas=(0.5, 0.999))
optD = optim.Adam(netD.parameters(), lr=D_LEARNING_RATE, betas=(0.5, 0.999))

# --- Loss functions ---
criterion_gan  = GANLoss(mode=GAN_MODE).to(device)
criterion_l1   = nn.L1Loss()
criterion_ssim = SSIMLoss().to(device)
criterion_perc = PerceptualLoss().to(device)

# --- Mixed precision scalers (separate for G and D) ---
scaler_G = GradScaler('cuda')
scaler_D = GradScaler('cuda')

# --- Resume from checkpoint if specified ---
START_EPOCH = 1
history = {
    'g_loss': [], 'd_loss': [],
    'l1': [], 'ssim': [], 'perceptual': [], 'adversarial': [],
    'epoch_time': [],
}
if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
    netG.load_state_dict(ckpt['netG_state_dict'])
    netD.load_state_dict(ckpt['netD_state_dict'])
    optG.load_state_dict(ckpt['optG_state_dict'])
    optD.load_state_dict(ckpt['optD_state_dict'])
    START_EPOCH = ckpt['epoch'] + 1
    if 'history' in ckpt:
        history = ckpt['history']
    print(f"Resumed from epoch {ckpt['epoch']} (G loss: {ckpt['g_loss']:.4f}, D loss: {ckpt['d_loss']:.4f})")
    print(f"Training will continue from epoch {START_EPOCH} to {NUM_EPOCHS}")
    del ckpt
    torch.cuda.empty_cache()
else:
    print('Starting fresh training.')

# --- Print model sizes ---
g_params = sum(p.numel() for p in netG.parameters() if p.requires_grad)
d_params = sum(p.numel() for p in netD.parameters() if p.requires_grad)
print(f'Generator params:     {g_params:,}')
print(f'Discriminator params: {d_params:,}')
print(f'Total trainable:      {g_params + d_params:,}')

# Sanity check: one forward pass
sample_input, sample_target, _ = next(iter(train_loader))
with torch.no_grad():
    g_out = netG(sample_input.to(device))
    d_out = netD(torch.cat([sample_input.to(device), g_out], dim=1))
print(f'G input:  {tuple(sample_input.shape)}  (RGB + outline)')
print(f'G output: {tuple(g_out.shape)}')
print(f'D output: {tuple(d_out.shape)}  (patch map)')
del g_out, d_out, sample_input, sample_target
torch.cuda.empty_cache()

In [ ]:
# 6. Training loop — alternating G/D updates, saves checkpoint + samples every N epochs

def save_training_samples(netG, dataloader, device, epoch, sample_dir, num_samples=4):
    """Save visual grid of (input, output, target) for tracking progress."""
    netG.eval()
    fig, axs = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    cols = ['Masked Input', 'Generator Output', 'Ground Truth']
    for j, title in enumerate(cols):
        axs[0, j].set_title(title, fontsize=14, fontweight='bold')
    for i in range(num_samples):
        inp, tgt, fname = dataloader.dataset[i]
        with torch.no_grad():
            out = netG(inp.unsqueeze(0).to(device))
        inp_np = inp[:3].permute(1, 2, 0).numpy()
        out_np = out.squeeze(0).cpu().permute(1, 2, 0).clamp(0, 1).numpy()
        tgt_np = tgt.permute(1, 2, 0).numpy()
        axs[i, 0].imshow(inp_np); axs[i, 0].set_ylabel(fname, fontsize=8)
        axs[i, 1].imshow(out_np)
        axs[i, 2].imshow(tgt_np)
        for j in range(3):
            axs[i, j].axis('off')
        del out
    plt.suptitle(f'Epoch {epoch}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(sample_dir, f'samples_epoch{epoch:04d}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    netG.train()

print(f'Training from epoch {START_EPOCH} to {NUM_EPOCHS}')
print('=' * 70)

for epoch in range(START_EPOCH, NUM_EPOCHS + 1):
    epoch_start = time.time()
    netG.train()
    netD.train()

    # Accumulators for per-epoch metrics
    run_g_loss = run_d_loss = 0.0
    run_l1 = run_ssim = run_perc = run_adv = 0.0
    n_batches = 0

    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for input_4ch, target, _ in progress:
        bs = input_4ch.size(0)
        input_4ch = input_4ch.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        # ===================== Train Discriminator =====================
        optD.zero_grad()
        with autocast('cuda'):
            # Real pair
            real_pair = torch.cat([input_4ch, target], dim=1)        # [B, 7, H, W]
            pred_real = netD(real_pair)
            loss_d_real = criterion_gan(pred_real, target_is_real=True)

            # Fake pair (detach generator output)
            fake_output = netG(input_4ch)
            fake_pair = torch.cat([input_4ch, fake_output.detach()], dim=1)
            pred_fake = netD(fake_pair)
            loss_d_fake = criterion_gan(pred_fake, target_is_real=False)

            loss_D = 0.5 * (loss_d_real + loss_d_fake)

        scaler_D.scale(loss_D).backward()
        scaler_D.step(optD)
        scaler_D.update()

        # ===================== Train Generator =====================
        optG.zero_grad()
        with autocast('cuda'):
            fake_output = netG(input_4ch)
            fake_pair = torch.cat([input_4ch, fake_output], dim=1)
            pred_fake = netD(fake_pair)

            # Individual losses
            loss_adv  = criterion_gan(pred_fake, target_is_real=True)
            loss_l1   = criterion_l1(fake_output, target)
            loss_ssim = criterion_ssim(fake_output, target)
            loss_perc = criterion_perc(fake_output, target)

            loss_G = (LAMBDA_ADV * loss_adv
                      + LAMBDA_L1 * loss_l1
                      + LAMBDA_SSIM * loss_ssim
                      + LAMBDA_PERCEPTUAL * loss_perc)

        scaler_G.scale(loss_G).backward()
        scaler_G.step(optG)
        scaler_G.update()

        # Accumulate
        run_d_loss += loss_D.item() * bs
        run_g_loss += loss_G.item() * bs
        run_l1     += loss_l1.item() * bs
        run_ssim   += loss_ssim.item() * bs
        run_perc   += loss_perc.item() * bs
        run_adv    += loss_adv.item() * bs
        n_batches  += bs

        progress.set_postfix({
            'D': f'{loss_D.item():.3f}',
            'G': f'{loss_G.item():.3f}',
            'L1': f'{loss_l1.item():.4f}',
        })

    # --- Epoch averages ---
    n = len(train_dataset)
    avg = {k: v / n for k, v in zip(
        ['d_loss', 'g_loss', 'l1', 'ssim', 'perceptual', 'adversarial'],
        [run_d_loss, run_g_loss, run_l1, run_ssim, run_perc, run_adv]
    )}
    epoch_time = time.time() - epoch_start
    avg['epoch_time'] = epoch_time

    for k, v in avg.items():
        history[k].append(v)

    print(f'Epoch {epoch}/{NUM_EPOCHS} [{epoch_time:.1f}s] | '
          f'D: {avg["d_loss"]:.4f}  G: {avg["g_loss"]:.4f}  '
          f'L1: {avg["l1"]:.4f}  SSIM: {avg["ssim"]:.4f}  '
          f'Perc: {avg["perceptual"]:.4f}  Adv: {avg["adversarial"]:.4f}')

    # --- Save checkpoint ---
    if epoch % SAVE_EVERY == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'ganA_phase1_epoch{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'netG_state_dict': netG.state_dict(),
            'netD_state_dict': netD.state_dict(),
            'optG_state_dict': optG.state_dict(),
            'optD_state_dict': optD.state_dict(),
            'g_loss': avg['g_loss'],
            'd_loss': avg['d_loss'],
            'history': history,
        }, ckpt_path)
        print(f'  >> Checkpoint saved: {ckpt_path}')

    # --- Save visual samples ---
    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        save_training_samples(netG, test_loader, device, epoch, SAMPLE_DIR)
        print(f'  >> Samples saved to {SAMPLE_DIR}')

# --- Save final model ---
final_path = os.path.join(CHECKPOINT_DIR, f'ganA_phase1_epoch{NUM_EPOCHS}_final.pth')
torch.save({
    'epoch': NUM_EPOCHS,
    'netG_state_dict': netG.state_dict(),
    'netD_state_dict': netD.state_dict(),
    'optG_state_dict': optG.state_dict(),
    'optD_state_dict': optD.state_dict(),
    'g_loss': history['g_loss'][-1],
    'd_loss': history['d_loss'][-1],
    'history': history,
}, final_path)
print(f'Final model saved: {final_path}')

# --- Save training history as JSON ---
log_path = os.path.join(LOG_DIR, 'ganA_phase1_history.json')
with open(log_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved: {log_path}')

# Free memory
del optG, optD, scaler_G, scaler_D
torch.cuda.empty_cache()
print(f'GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# 7. Plot training metrics — comprehensive multi-panel view

epochs = range(1, len(history['g_loss']) + 1)

fig, axs = plt.subplots(2, 3, figsize=(18, 10))

# G loss
axs[0, 0].plot(epochs, history['g_loss'], 'b-', linewidth=1.5)
axs[0, 0].set_title('Generator Loss (total)')
axs[0, 0].set_xlabel('Epoch'); axs[0, 0].set_ylabel('Loss'); axs[0, 0].grid(True)

# D loss
axs[0, 1].plot(epochs, history['d_loss'], 'r-', linewidth=1.5)
axs[0, 1].set_title('Discriminator Loss')
axs[0, 1].set_xlabel('Epoch'); axs[0, 1].set_ylabel('Loss'); axs[0, 1].grid(True)

# L1
axs[0, 2].plot(epochs, history['l1'], 'g-', linewidth=1.5)
axs[0, 2].set_title('L1 Reconstruction Loss')
axs[0, 2].set_xlabel('Epoch'); axs[0, 2].set_ylabel('L1'); axs[0, 2].grid(True)

# SSIM loss (lower = better)
axs[1, 0].plot(epochs, history['ssim'], 'm-', linewidth=1.5)
axs[1, 0].set_title('SSIM Loss (1 - SSIM)')
axs[1, 0].set_xlabel('Epoch'); axs[1, 0].set_ylabel('SSIM Loss'); axs[1, 0].grid(True)

# Perceptual
axs[1, 1].plot(epochs, history['perceptual'], 'c-', linewidth=1.5)
axs[1, 1].set_title('Perceptual Loss (VGG)')
axs[1, 1].set_xlabel('Epoch'); axs[1, 1].set_ylabel('Loss'); axs[1, 1].grid(True)

# Adversarial
axs[1, 2].plot(epochs, history['adversarial'], color='orange', linewidth=1.5)
axs[1, 2].set_title('Adversarial Loss (G)')
axs[1, 2].set_xlabel('Epoch'); axs[1, 2].set_ylabel('Loss'); axs[1, 2].grid(True)

plt.suptitle('GAN-A Phase 1 Training Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'ganA_phase1_loss_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# Also show D vs G balance
fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(epochs, history['g_loss'], 'b-', label='G Loss', linewidth=1.5)
ax2.plot(epochs, history['d_loss'], 'r-', label='D Loss', linewidth=1.5)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.set_title('Generator vs Discriminator Loss Balance')
ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'ganA_phase1_gd_balance.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print(f"Final metrics (epoch {len(history['g_loss'])}):")
for k in ['g_loss', 'd_loss', 'l1', 'ssim', 'perceptual', 'adversarial']:
    print(f'  {k:15s}: {history[k][-1]:.6f}')
print(f'  {"SSIM value":15s}: {1 - history["ssim"][-1]:.6f}')
total_time = sum(history['epoch_time'])
print(f'  Total training time: {total_time/3600:.2f} hours')

# Test Evaluation

Use the cells below to test your trained model on any image from the test dataset.

In [ ]:
# 8. (Optional) Load a specific checkpoint for testing
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'ganA_phase1_epoch100_final.pth'), map_location=device, weights_only=False)
netG.load_state_dict(ckpt['netG_state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']}")

# List available test images
test_files = sorted(os.listdir(TEST_IMAGE_DIR))
print(f'{len(test_files)} test images available:')
for i, f in enumerate(test_files):
    print(f'  [{i}] {f}')

In [ ]:
# 9. Test on a specific image — change TEST_INDEX to try different images
TEST_INDEX = 5  # <-- Change this to test any image (0 to len-1)

torch.cuda.empty_cache()
netG.eval()
input_4ch, target_tensor, fname = test_dataset[TEST_INDEX]

with torch.no_grad():
    output = netG(input_4ch.unsqueeze(0).to(device))
output_np = output.squeeze(0).cpu().permute(1, 2, 0).clamp(0, 1).numpy()
input_np = input_4ch[:3].permute(1, 2, 0).numpy()  # show only RGB channels
target_np = target_tensor.permute(1, 2, 0).numpy()

# Compute metrics
l1_err = np.mean(np.abs(output_np - target_np))
mse = np.mean((output_np - target_np) ** 2)
psnr = 10 * np.log10(1.0 / (mse + 1e-10))
with torch.no_grad():
    # Ensure both tensors are on the same device as criterion_ssim
    ssim_val = criterion_ssim.ssim_value(output, target_tensor.unsqueeze(0).to(output.device)).item()
del output

# Difference map (amplified for visibility)
diff_map = np.mean(np.abs(output_np - target_np), axis=2)

fig, axs = plt.subplots(1, 4, figsize=(20, 5))
axs[0].imshow(input_np)
axs[0].set_title(f'Masked Input\n{fname}')
axs[0].axis('off')
axs[1].imshow(output_np)
axs[1].set_title(f'Generator Output\nL1: {l1_err:.4f} | PSNR: {psnr:.2f} dB')
axs[1].axis('off')
axs[2].imshow(target_np)
axs[2].set_title(f'Ground Truth\nSSIM: {ssim_val:.4f}')
axs[2].axis('off')
im = axs[3].imshow(diff_map, cmap='hot', vmin=0, vmax=0.3)
axs[3].set_title('Error Map (|output - target|)')
axs[3].axis('off')
plt.colorbar(im, ax=axs[3], fraction=0.046)
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, f'test_sample_{TEST_INDEX}.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 10. Test on a specific image by file path (requires outline path too)
# Update these paths to match your cloud filesystem
TEST_IMAGE_PATH   = os.path.join(str(repo_root), 'data', 'masks', 'lesion_synthetic', 'mask_healthy_0001.png')  # <-- Change this
TEST_OUTLINE_PATH = os.path.join(str(repo_root), 'data', 'gan_a', 'phase_1', 'test', 'outline', 'healthy_0001.png')  # <-- Change this

torch.cuda.empty_cache()
netG.eval()

img = Image.open(TEST_IMAGE_PATH).convert('RGB')
outline = Image.open(TEST_OUTLINE_PATH).convert('L')
tfm_rgb = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])
tfm_gray = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])
img_t = tfm_rgb(img)
outline_t = tfm_gray(outline)
input_4ch = torch.cat([img_t, outline_t], dim=0)  # [4, H, W]

with torch.no_grad():
    output = netG(input_4ch.unsqueeze(0).to(device))
output_np = output.squeeze(0).cpu().permute(1, 2, 0).clamp(0, 1).numpy()
input_np = img_t.permute(1, 2, 0).numpy()
del output

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].imshow(input_np)
axs[0].set_title(f'Input\n{os.path.basename(TEST_IMAGE_PATH)}')
axs[0].axis('off')
axs[1].imshow(output_np)
axs[1].set_title('Generator Output')
axs[1].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'test_custom_path.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 11. Test on a GLS image with real lesion mask overlay (mask turned black)
GLS_IMAGE_DIR   = os.path.join(str(repo_root), 'data', 'diseased', 'gls')
GLS_MASK_DIR    = os.path.join(str(repo_root), 'data', 'masks', 'lesion_real', 'gls')
GLS_OUTLINE_DIR = os.path.join(str(repo_root), 'data', 'masks', 'leaf')  # leaf outlines for GLS images
GLS_INDEX = 10  # <-- Change this to test different images

# List available GLS images
image_files = sorted([f for f in os.listdir(GLS_IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
mask_files = sorted([f for f in os.listdir(GLS_MASK_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
outline_files = sorted([f for f in os.listdir(GLS_OUTLINE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

img_path = os.path.join(GLS_IMAGE_DIR, image_files[GLS_INDEX])
mask_path = os.path.join(GLS_MASK_DIR, mask_files[GLS_INDEX])
outline_path = os.path.join(GLS_OUTLINE_DIR, outline_files[GLS_INDEX])

img = Image.open(img_path).convert('RGB')
mask = Image.open(mask_path).convert('RGB')
outline = Image.open(outline_path).convert('L')

# Turn mask black
black_mask = Image.new('RGB', mask.size, (0, 0, 0))
img_with_mask = Image.composite(black_mask, img, mask.convert('L').point(lambda x: 255 if x > 0 else 0))

# Preprocess for model
tfm_rgb = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])
tfm_gray = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])
img_t = tfm_rgb(img_with_mask)
outline_t = tfm_gray(outline)
input_4ch = torch.cat([img_t, outline_t], dim=0)  # [4, H, W]

# Run through model
netG.eval()
torch.cuda.empty_cache()
with torch.no_grad():
    output = netG(input_4ch.unsqueeze(0).to(device))
output_np = output.squeeze(0).cpu().permute(1, 2, 0).clamp(0, 1).numpy()
input_np = img_t.permute(1, 2, 0).numpy()
original_np = tfm_rgb(img).permute(1, 2, 0).numpy()
del output

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].imshow(input_np)
axs[0].set_title(f'GLS Input w/ Black Mask\n{os.path.basename(img_path)}')
axs[0].axis('off')
axs[1].imshow(output_np)
axs[1].set_title('Generator Output (Inpainted)')
axs[1].axis('off')
axs[2].imshow(original_np)
axs[2].set_title('Original Diseased (Reference)')
axs[2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, f'test_gls_{GLS_INDEX}.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 12. Evaluate on FULL test set — compute average L1, SSIM, PSNR
torch.cuda.empty_cache()
netG.eval()
test_l1 = 0.0
test_ssim = 0.0
test_psnr = 0.0
n_test = 0

with torch.no_grad():
    for input_4ch, target, _ in tqdm(test_loader, desc='Evaluating test set'):
        input_4ch = input_4ch.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        out = netG(input_4ch)

        # L1
        test_l1 += criterion_l1(out, target).item()

        # SSIM
        test_ssim += criterion_ssim.ssim_value(out, target).item()

        # PSNR
        mse_val = torch.mean((out - target) ** 2).item()
        test_psnr += 10 * np.log10(1.0 / (mse_val + 1e-10))

        n_test += 1
        del out

avg_l1   = test_l1 / n_test
avg_ssim = test_ssim / n_test
avg_psnr = test_psnr / n_test

print('=' * 50)
print('FULL TEST SET EVALUATION')
print('=' * 50)
print(f'  Average L1 Loss:  {avg_l1:.6f}')
print(f'  Average SSIM:     {avg_ssim:.6f}')
print(f'  Average PSNR:     {avg_psnr:.2f} dB')
print('=' * 50)